# Logistic regression model

Logistic regression works completely differently from trees. It gives each feature one weight, called a **coefficient**, and adds them up:

`risk score = w1 × feature1 + w2 × feature2 + ... + intercept`

Then it squashes that total into a probability from 0 to 1.

That makes it simple and stable, but it can only learn **straight-line effects**: more of a feature always means proportionally more or less risk. It can't learn "risk is high for the youngest and oldest" or "this matters only for renters" unless you build those features yourself.

That's why it's usually weaker alone, but its mistakes differ from the trees' mistakes, so it may still help the blend.

## 0. Reload shared code automatically

Your features live in `features.py`, which every model notebook shares. Python normally loads a file like that only once per kernel, so later edits would be ignored.

`autoreload` makes the notebook pick up your latest `features.py` each time a cell runs. Keep this as the first code cell.

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Project folders

This notebook lives in `notebooks/`, but the data, shared code and outputs are in other folders. This cell finds the project folder and imports their locations from `scripts/paths.py`:

| Name | Folder | Holds |
|---|---|---|
| `DATA` | `data/` | Competition files |
| `PREDS` | `preds_for_blend/` | Saved predictions for blending |
| `SUBMISSIONS` | `submissions/` | Files to upload |
| `LOGS` | `logs/` | CatBoost training logs |

It also adds `scripts/` to Python's search path, so `from features import ...` works.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from paths import DATA, LOGS, PREDS, SUBMISSIONS

## 1. Imports

The scikit-learn tools here prepare the data inside a **pipeline**, explained in step 4.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from features import add_features

## 2. Settings

`SEED = 42` and 5 folds match your other notebooks. That's essential: the blend only works if every model is tested on **exactly the same folds**, so their out-of-fold predictions are comparable.

In [12]:
TARGET = "12-Month Default Flag"
SEED = 42
N_SPLITS = 5
PRED_DIR = PREDS
PRED_DIR.mkdir(exist_ok=True)

## 3. Load the data

The same as in your CatBoost notebook: `X` has the features, `y` has the 0/1 flag, and `X_test` has the same columns in the same order.

Never sort or filter rows here. The saved predictions must stay in the original row order, or the blend would line up the wrong people.

**Features:** `add_features()` from `features.py` adds your engineered columns. It runs on train and test in exactly the same way, so they always have the same columns. Here, `X_test` takes `X`'s column list, which keeps the order identical.


In [13]:
train = pd.read_csv(DATA / "train_dataset.csv")
test = pd.read_csv(DATA / "test_without_gt.csv")

X = add_features(train.drop(columns=["ID", TARGET]))
y = train[TARGET]
X_test = add_features(test)[X.columns]

cats = [
    "Application Channel", "Occupation Category", "Industry Category",
    "Employment Type Category", "Insurance Type", "Company Size Category",
    "Business Entity Type", "Residence Type", "Residence Ownership Status",
]
nums = [c for c in X.columns if c not in cats]
print(X.shape, X_test.shape)

(68546, 26) (137091, 26)


## 4. Build the model as a pipeline

Unlike trees, logistic regression needs its data prepared:

| Step | Applies to | Why |
|---|---|---|
| `OneHotEncoder` | Category codes | Gives each code its own 0/1 column, so occupation 1540 isn't treated as "more" than 140. `handle_unknown="ignore"` means a code never seen in training just gets all zeros, rather than an error. |
| `np.log1p` | Numbers | Compresses huge values, such as an income of ¥94.8 billion, so they can't dominate. `log1p` means log(1 + x), which works for zeros. `feature_names_out="one-to-one"` keeps the column names, so we can read the coefficients later. |
| `StandardScaler` | Numbers | Puts every numeric column on a similar scale, so regularization treats them evenly. |
| `LogisticRegression(C=0.1)` | Everything | `C` controls regularization: smaller means stronger, which keeps coefficients small and reduces overfitting. |

**Why a pipeline matters:** the scaler learns each column's mean and spread from data. Inside a pipeline, it's refitted on each training fold only, so the validation fold can't leak into it.

In [31]:
def make_logreg():
    numeric = make_pipeline(
        FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()
    )
    prep = make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), cats),
        (numeric, nums),
    )
    return make_pipeline(prep, LogisticRegression(C=0.01, max_iter=3000))

## 5. Cross-validate

This is your original `evaluate()` loop. There's no early stopping, because logistic regression doesn't build trees.

In [35]:
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
scores = []

for fold, (fit_idx, val_idx) in enumerate(cv.split(X, y)):
    model = make_logreg()
    model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
    oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    scores.append(roc_auc_score(y.iloc[val_idx], oof[val_idx]))
    print(f"Fold {fold}: AUC {scores[-1]:.4f}")

print(f"\nLogistic regression CV AUC: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

Fold 0: AUC 0.6290
Fold 1: AUC 0.6432
Fold 2: AUC 0.6364
Fold 3: AUC 0.6357
Fold 4: AUC 0.6329

Logistic regression CV AUC: 0.6354 +/- 0.0047


## 6. Train the final model and read its coefficients

The final model trains on all the data. Its coefficients show how each feature pushes the score:
- **Positive:** higher values mean higher risk
- **Negative:** higher values mean lower risk

The numeric columns are scaled, so their coefficients are roughly comparable with each other. Category columns appear as one coefficient per code, such as `onehotencoder__Residence Type_4`.

These show association, not cause. Related features, such as age and residence period, also share credit.

In [36]:
final_model = make_logreg().fit(X, y)
test_pred = final_model.predict_proba(X_test)[:, 1]

names = final_model[0].get_feature_names_out()
coefs = pd.Series(final_model[-1].coef_[0], index=names).sort_values()
print("Lowest risk:\n", coefs.head(8).round(3))
print("\nHighest risk:\n", coefs.tail(8).round(3))

Lowest risk:
 onehotencoder__Company Size Category_1    -0.207
pipeline__Age (in months)                 -0.195
onehotencoder__Industry Category_2        -0.184
onehotencoder__Company Size Category_2    -0.184
pipeline__Declared Borrowing Amount       -0.160
onehotencoder__Occupation Category_1140   -0.160
onehotencoder__Application Channel_20     -0.150
onehotencoder__Industry Category_16       -0.139
dtype: float64

Highest risk:
 onehotencoder__Industry Category_5           0.123
onehotencoder__Occupation Category_740       0.127
pipeline__Desired Credit Limit               0.146
onehotencoder__Application Channel_17        0.152
onehotencoder__Occupation Category_1540      0.156
onehotencoder__Company Size Category_7       0.197
onehotencoder__Employment Type Category_2    0.266
pipeline__Declared Number of Borrowings      0.340
dtype: float64


## Write the submission

The `assert` lines stop the notebook if the file breaks the competition's format, so you don't waste one of your 10 daily submissions on a bad upload.

In [37]:
sub = pd.read_csv(DATA / "submission_template_for_test.csv")
assert (sub["ID"].values == test["ID"].values).all(), "Row order doesn't match the template"

sub[TARGET] = test_pred

assert list(sub.columns) == ["ID", TARGET]
assert len(sub) == 137_091
assert sub[TARGET].between(0, 1).all() and sub[TARGET].notna().all()

sub.to_csv(SUBMISSIONS / "submission_logreg.csv", index=False)
print("Saved submissions/submission_logreg.csv")

Saved submission_logreg.csv


## Save predictions for the blend

This is what `06_blend.ipynb` reads. It needs both files:
- `logreg_oof.npy`: one honest prediction per **training** row, used to score the blend
- `logreg_test.npy`: one prediction per **test** row, used to build the blended submission

These are derived from competition data, so delete the `preds_for_blend` folder after the competition.

In [ ]:
np.save(PRED_DIR / "logreg_oof.npy", oof)
np.save(PRED_DIR / "logreg_test.npy", test_pred)
print("Saved preds_for_blend/logreg_oof.npy and preds_for_blend/logreg_test.npy")

Saved preds_for_blend/logreg_oof.npy and preds_for_blend/logreg_test.npy
